In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip install -q tensorflow tensorflow-recommenders


In [ ]:
# ===== MUST BE FIRST CELL =====
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip uninstall -y keras tensorflow tensorflow-recommenders
!pip install tensorflow tensorflow-recommenders pandas

print("✅ Restart runtime after this cell")


Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
Found existing installation: tensorflow 2.19.1
Uninstalling tensorflow-2.19.1:
  Successfully uninstalled tensorflow-2.19.1
Found existing installation: tensorflow-recommenders 0.7.7
Uninstalling tensorflow-recommenders-0.7.7:
  Successfully uninstalled tensorflow-recommenders-0.7.7
  Using cached tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.5 kB)
  Using cached tensorflow_recommenders-0.7.7-py3-none-any.whl.metadata (4.6 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.13.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
Using cached tensorflow_recommenders-0.7.7-py3-none-any.whl (108 kB)
Using cached keras-3.13.2-py3-none-any.whl (1.5 MB)
Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/ncf_project_new"

os.makedirs(f"{BASE_DIR}/training", exist_ok=True)
os.makedirs(f"{BASE_DIR}/models/ncf_model", exist_ok=True)

print("Project folders created")

Project folders created


In [ ]:
# -----------------------
# Load dataset
# -----------------------
import pandas as pd

# Load CSV file
df = pd.read_csv('/content/drive/MyDrive/datasets/last_merge_new.csv')

# Sort by timestamp
df = df.sort_values("timestamp").reset_index(drop=True)

# Select only needed columns
df = df[['user_id', 'behavior_type', 'timestamp', 'item_id', 'category_id']]

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_recommenders as tfrs
import pickle
from typing import Dict, Text

print("✅ TensorFlow version:", tf.__version__)
print("✅ TFRS imported successfully")


✅ TensorFlow version: 2.19.1
✅ TFRS imported successfully


In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_recommenders as tfrs
from typing import Dict, Text

# -----------------------
# Split train/test
# -----------------------
train_df = df.iloc[:1_800_000].copy()
test_df  = df.iloc[1_800_000:].copy()

# Map behavior type to numeric weight
behavior_map = {"pv": 1.0, "fav": 3.0, "cart": 5.0, "buy": 10.0}
train_df["weight"] = train_df["behavior_type"].map(behavior_map).astype(float)
test_df["weight"]  = test_df["behavior_type"].map(behavior_map).astype(float)

# Ensure string type
for col in ["user_id", "item_id", "category_id"]:
    train_df[col] = train_df[col].astype(str)
    test_df[col]  = test_df[col].astype(str)

# Save test set
test_df.to_csv("test_set_200k_new.csv", index=False)

# -----------------------
# Split train into chunks
# -----------------------
chunk_size = 100_000
num_chunks = int(np.ceil(len(train_df) / chunk_size))

for i in range(num_chunks):
    train_df.iloc[i*chunk_size:(i+1)*chunk_size] \
        .to_csv(f"train_chunk_{i+1}.csv", index=False)

# -----------------------
# Vocabulary
# -----------------------
unique_users = train_df["user_id"].unique().tolist()
unique_items = train_df["item_id"].unique().tolist()

# -----------------------
# NCF Model
# -----------------------
class NCFModel(tf.keras.Model):
    def __init__(self, unique_users, unique_items, embedding_dim=32):
        super().__init__()

        self.user_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary=unique_users, mask_token=None
            ),
            tf.keras.layers.Embedding(len(unique_users) + 1, embedding_dim)
        ])

        self.item_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary=unique_items, mask_token=None
            ),
            tf.keras.layers.Embedding(len(unique_items) + 1, embedding_dim)
        ])

        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(1)
        ])

    def call(self, inputs, training=False):
        user_emb = self.user_embedding(inputs["user_id"])
        item_emb = self.item_embedding(inputs["item_id"])
        x = tf.concat([user_emb, item_emb], axis=1)
        return self.mlp(x)

# -----------------------
# TFRS Wrapper
# -----------------------
class NCFRecommender(tfrs.models.Model):
    def __init__(self, unique_users, unique_items):
        super().__init__()
        self.ncf = NCFModel(unique_users, unique_items)
        self.task = tfrs.tasks.Ranking(
            loss=tf.keras.losses.MeanSquaredError(),
            metrics=[tf.keras.metrics.RootMeanSquaredError()]
        )

    def compute_loss(
        self, features: Dict[Text, tf.Tensor], training=False
    ) -> tf.Tensor:
        labels = features["weight"]
        predictions = self.ncf(
            {
                "user_id": features["user_id"],
                "item_id": features["item_id"]
            },
            training=training
        )
        return self.task(labels=labels, predictions=predictions)

# -----------------------
# Initialize
# -----------------------
model = NCFRecommender(unique_users, unique_items)
model.compile(optimizer=tf.keras.optimizers.Adam(0.001))

In [ ]:
# -----------------------
# Chunk-by-chunk training
# -----------------------
for i in range(1, num_chunks + 1):
    print(f"\n=== Training chunk {i}/{num_chunks} ===")

    chunk = pd.read_csv(f"train_chunk_{i}.csv")

    # Ensure correct type
    for col in ["user_id", "item_id"]:
        chunk[col] = chunk[col].astype(str)

    ds = tf.data.Dataset.from_tensor_slices({
        "user_id": chunk["user_id"].values,
        "item_id": chunk["item_id"].values,
        "weight": chunk["weight"].values.astype(np.float32)
    })

    ds = ds.shuffle(50_000).batch(2048).prefetch(tf.data.AUTOTUNE)

    model.fit(ds, epochs=3 if i == 1 else 1, verbose=1)


# -----------------------
# Evaluate
# -----------------------
test_df = pd.read_csv("test_set_200k_new.csv")

test_ds = tf.data.Dataset.from_tensor_slices({
    "user_id": test_df["user_id"].astype(str).values,
    "item_id": test_df["item_id"].astype(str).values,
    "weight": test_df["weight"].astype(np.float32).values
}).batch(2048).prefetch(tf.data.AUTOTUNE)

results = model.evaluate(test_ds, verbose=1)
print("\n✅ Test results:", results)


=== Training chunk 1/18 ===
Epoch 1/3
49/49 [==============================] - 5s 69ms/step - root_mean_squared_error: 1.9277 - loss: 3.6910 - regularization_loss: 0.0000e+00 - total_loss: 3.6910
Epoch 2/3
49/49 [==============================] - 3s 51ms/step - root_mean_squared_error: 1.5653 - loss: 2.4449 - regularization_loss: 0.0000e+00 - total_loss: 2.4449
Epoch 3/3
49/49 [==============================] - 2s 38ms/step - root_mean_squared_error: 1.4939 - loss: 2.2351 - regularization_loss: 0.0000e+00 - total_loss: 2.2351

=== Training chunk 2/18 ===
49/49 [==============================] - 2s 38ms/step - root_mean_squared_error: 1.5073 - loss: 2.2678 - regularization_loss: 0.0000e+00 - total_loss: 2.2678

=== Training chunk 3/18 ===
49/49 [==============================] - 2s 38ms/step - root_mean_squared_error: 1.5013 - loss: 2.2586 - regularization_loss: 0.0000e+00 - total_loss: 2.2586

=== Training chunk 4/18 ===
49/49 [==============================] - 2s 40ms/step - root_mea

In [ ]:
import tensorflow as tf
import os

EXPORT_DIR = f"{BASE_DIR}/models/ncf_savedmodel_new/1"

# Clean old export (important)
import shutil
if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)

class NCFServing(tf.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.model = trained_model.ncf  # your NCFModel

    @tf.function(input_signature=[
        tf.TensorSpec(shape=[None], dtype=tf.string, name="user_id"),
        tf.TensorSpec(shape=[None], dtype=tf.string, name="item_id"),
    ])
    def serving_default(self, user_id, item_id):
        scores = self.model({"user_id": user_id, "item_id": item_id})  # (batch,1)
        return {"score": tf.squeeze(scores, axis=1)}  # (batch,)

serving = NCFServing(model)

tf.saved_model.save(
    serving,
    EXPORT_DIR,
    signatures={"serving_default": serving.serving_default}
)

print("✅ Re-exported SavedModel with named inputs to:", EXPORT_DIR)


✅ Re-exported SavedModel with named inputs to: /content/drive/MyDrive/ncf_project_new/models/ncf_savedmodel_new/1


In [ ]:
import shutil

ZIP_PATH = f"{BASE_DIR}/models/ncf_savedmodel_new.zip"
shutil.make_archive(ZIP_PATH.replace(".zip",""), "zip", f"{BASE_DIR}/models/ncf_savedmodel_new")
print("✅ Zipped to:", ZIP_PATH)


✅ Zipped to: /content/drive/MyDrive/ncf_project_new/models/ncf_savedmodel_new.zip
